# Assignment 5.1 — ML System Observability: Model Bias Monitor

**Course:** AAI-540 · **Author:** Sourangshu Pal (Group 1)

Lab 5.1 implements a **Model Quality Monitor** for the XGBoost churn-prediction endpoint. This assignment adds the second pillar of observability: a SageMaker Clarify **Model Bias Monitor**, which watches live predictions for bias drift across a sensitive facet (here: account tenure) and raises CloudWatch alerts when bias metrics leave their baseline bounds.

**Lineage:** Lab 5.1 (`01_model_quality_monitor.ipynb`) + the AWS example *Monitoring bias drift and feature attribution drift* (amazon-sagemaker-examples). The monitor's analysis results are also visualized in SageMaker Studio under the endpoint's monitoring tab — either view is acceptable for the required screenshot.

**Deliverables:** 📸 screenshot of the monitor's report output, then submit `Sourangshu_Pal_Assignment5.1.ipynb` + `Sourangshu_Pal_Assignment5.1.pdf` (export via `jupyter nbconvert --to html` → print to PDF).

---
## Section 1 — Setup

Same session/role/bucket wiring as Lab 5.1. All captured data, ground truth, baselines, and reports live under one prefix in the default bucket.

In [ ]:
import json
import random
import threading
import time

import pandas as pd
from datetime import datetime, timedelta

from sagemaker import get_execution_role, image_uris, Session
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig,
)
from sagemaker.model import Model
from sagemaker.model_monitor import (
    CronExpressionGenerator,
    DataCaptureConfig,
    EndpointInput,
    ModelBiasMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.s3 import S3Downloader, S3Uploader

role = get_execution_role()
sagemaker_session = Session()
sagemaker_client = sagemaker_session.sagemaker_client
sagemaker_runtime_client = sagemaker_session.sagemaker_runtime_client

region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()

prefix = "sagemaker/Churn-ModelBiasMonitor-assignment-5-1"
s3_key = f"s3://{bucket}/{prefix}"
s3_capture_upload_path = f"{s3_key}/datacapture"
ground_truth_upload_path = f"{s3_key}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
s3_report_path = f"{s3_key}/reports"
baseline_results_uri = f"{s3_key}/baselining"

dataset_type = "text/csv"
endpoint_instance_count = 1
endpoint_instance_type = "ml.m5.large"
schedule_expression = CronExpressionGenerator.hourly()

print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

---
## Section 2 — Deploy the churn model with data capture enabled

Pre-trained XGBoost churn artifact (same one as Lab 5.1), deployed to a real-time endpoint with 100% data capture so the monitor has inference data to analyze.

In [ ]:
model_url = S3Uploader.upload("model/xgb-churn-prediction-model.tar.gz", s3_key)
print(f"Model uploaded to {model_url}")

model_name = f"assignment51-xgb-churn-pred-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
endpoint_name = f"assignment51-xgb-churn-model-bias-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print(f"Model: {model_name}\nEndpoint: {endpoint_name}")

image_uri = image_uris.retrieve("xgboost", region, "0.90-1")
model = Model(
    role=role,
    name=model_name,
    image_uri=image_uri,
    model_data=model_url,
    sagemaker_session=sagemaker_session,
)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=s3_capture_upload_path,
)
model.deploy(
    initial_instance_count=endpoint_instance_count,
    instance_type=endpoint_instance_type,
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
)

---
## Section 3 — Generate traffic and ground truth

A worker thread keeps invoking the endpoint (a unique `InferenceId` per row joins captures to ground truth), and a second thread uploads synthetic ground-truth labels hourly in the merge-container JSONL format. Without steady traffic **and** ground truth, monitoring executions fail for lack of data — same requirement as Lab 5.1.

In [ ]:
class WorkerThread(threading.Thread):
    def __init__(self, do_run, *args, **kwargs):
        super(WorkerThread, self).__init__(*args, **kwargs)
        self.__do_run = do_run
        self.__terminate_event = threading.Event()

    def terminate(self):
        self.__terminate_event.set()

    def run(self):
        while not self.__terminate_event.is_set():
            self.__do_run(self.__terminate_event)


test_dataset = "test_data/test-dataset-input-cols.csv"


def invoke_endpoint(terminate_event):
    with open(test_dataset, "r") as f:
        i = 0
        for row in f:
            payload = row.rstrip("\n")
            sagemaker_runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Body=payload,
                InferenceId=str(i),  # joins captures to ground truth
            )
            i += 1
            time.sleep(1)
            if terminate_event.is_set():
                break


invoke_endpoint_thread = WorkerThread(do_run=invoke_endpoint)
invoke_endpoint_thread.start()

In [ ]:
def ground_truth_with_id(inference_id):
    random.seed(inference_id)  # consistent results per id
    rand = random.random()
    return {
        "groundTruthData": {
            "data": "1" if rand < 0.7 else "0",  # ~70% positive labels
            "encoding": "CSV",
        },
        "eventMetadata": {"eventId": str(inference_id)},
        "eventVersion": "0",
    }


def upload_ground_truth(upload_time):
    records = [ground_truth_with_id(i) for i in range(test_dataset_size)]
    fake_records = [json.dumps(r) for r in records]
    data_to_upload = "\n".join(fake_records)
    target_s3_uri = f"{ground_truth_upload_path}/{upload_time:%Y/%m/%d/%H/%M%S}.jsonl"
    print(f"Uploading {len(fake_records)} records to", target_s3_uri)
    S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)


# count rows sent for inference (same file the traffic thread reads)
with open(test_dataset, "r") as f:
    test_dataset_size = sum(1 for _ in f)
print(f"test dataset rows: {test_dataset_size}")

# ground truth for the last hour (so the first execution has data to merge)
upload_ground_truth(datetime.utcnow() - timedelta(hours=1))


def generate_fake_ground_truth(terminate_event):
    upload_ground_truth(datetime.utcnow())
    for _ in range(0, 60):
        time.sleep(60)
        if terminate_event.is_set():
            break


ground_truth_thread = WorkerThread(do_run=generate_fake_ground_truth)
ground_truth_thread.start()

---
## Section 4 — Model Bias Monitor: baselining job

The baselining job runs predictions over the labeled validation set through a *shadow endpoint* and suggests bias-metric constraints (e.g. allowed ranges for DPPL, DI, etc.) that the scheduled monitor will enforce.

**Headers / facet choice.** The lab's data is anonymized: `test-dataset-input-cols.csv` names the 69 features `col1..col69`, and `validation.csv` is headerless with the label in the first field. We build a headered validation file (`Churn` + the lab's own column names) and facet on **`col1`** — the field right after the label, i.e. **Account Length** in the original churn schema (observed range 1–225, mean ≈ 101, so both facet groups are well populated) — with threshold 100, mirroring the AWS documentation example (`facet_name="Account Length"`, `facet_values_or_threshold=[100]`). The predicted label is derived from the model's churn probability with the same 0.8 cutoff used in Lab 5.1.

In [ ]:
# Build a headered validation CSV: 'Churn' + the lab's own feature headers
validation_raw = "test_data/validation.csv"
validation_dataset = "test_data/validation-dataset-with-header.csv"

with open(test_dataset, "r") as f:
    all_headers = f.readline().rstrip().split(",")  # col1..col69
label_header = "Churn"
all_headers = [label_header] + all_headers
print(f"{len(all_headers)} headers, first five: {all_headers[:5]}")

with open(validation_dataset, "w") as out:
    out.write(",".join(all_headers) + "\n")
    with open(validation_raw, "r") as f:
        for row in f:
            out.write(row)

model_bias_monitor = ModelBiasMonitor(
    role=role,
    sagemaker_session=sagemaker_session,
    max_runtime_in_seconds=1800,
)

model_bias_baselining_job_result_uri = f"{baseline_results_uri}/model_bias"
model_bias_data_config = DataConfig(
    s3_data_input_path=validation_dataset,
    s3_output_path=model_bias_baselining_job_result_uri,
    label=label_header,
    headers=all_headers,
    dataset_type=dataset_type,
)

model_bias_config = BiasConfig(
    label_values_or_threshold=[1],
    facet_name="col1",  # first feature = Account Length (range 1-225, mean ~101)
    facet_values_or_threshold=[100],
)

model_predicted_label_config = ModelPredictedLabelConfig(
    probability_threshold=0.8,
)

model_config = ModelConfig(
    model_name=model_name,
    instance_count=endpoint_instance_count,
    instance_type=endpoint_instance_type,
    content_type=dataset_type,
    accept_type=dataset_type,
)

In [ ]:
model_bias_monitor.suggest_baseline(
    model_config=model_config,
    data_config=model_bias_data_config,
    bias_config=model_bias_config,
    model_predicted_label_config=model_predicted_label_config,
)
print(f"ModelBiasMonitor baselining job: {model_bias_monitor.latest_baselining_job_name}")

In [ ]:
model_bias_monitor.latest_baselining_job.wait(logs=False)
model_bias_constraints = model_bias_monitor.suggested_constraints()
print(f"Suggested constraints: {model_bias_constraints.file_s3_uri}")
print(S3Downloader.read_file(model_bias_constraints.file_s3_uri))

---
## Section 5 — Schedule the bias drift monitor

Hourly schedule over the endpoint's captured data (1-hour lookback window), merged with ground truth by the first-party merge container. Because a baselining job was submitted, the schedule automatically picks up its analysis configuration. `probability_threshold_attribute=0.8` matches the baseline cutoff.

In [ ]:
model_bias_monitor.create_monitoring_schedule(
    output_s3_uri=s3_report_path,
    endpoint_input=EndpointInput(
        endpoint_name=endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        start_time_offset="-PT1H",
        end_time_offset="-PT0H",
        probability_threshold_attribute=0.8,
    ),
    ground_truth_input=ground_truth_upload_path,
    schedule_cron_expression=schedule_expression,
    enable_cloudwatch_metrics=True,
)
print(f"Model bias monitoring schedule: {model_bias_monitor.monitoring_schedule_name}")

---
## Section 6 — Wait for the first execution and inspect the report

An hourly schedule fires at the top of the hour plus an AWS buffer of 0–20 minutes, so the first execution can take up to ~80 minutes to appear. Keep the notebook (and the Learner Lab) running; the cells below poll until the execution starts, then we stop the schedule to avoid extra charges while letting the in-flight execution finish.

In [ ]:
def wait_for_execution_to_start(model_monitor):
    print("An hourly schedule kicks off executions ON the hour (plus 0-20 min buffer).")
    print("Waiting for the first execution to happen", end="")
    schedule_desc = model_monitor.describe_schedule()
    while "LastMonitoringExecutionSummary" not in schedule_desc:
        schedule_desc = model_monitor.describe_schedule()
        print(".", end="", flush=True)
        time.sleep(60)
    print("\nDone! Execution has been created")
    print("Now waiting for execution to start", end="")
    while schedule_desc["LastMonitoringExecutionSummary"]["MonitoringExecutionStatus"] in "Pending":
        schedule_desc = model_monitor.describe_schedule()
        print(".", end="", flush=True)
        time.sleep(10)
    print("\nDone! Execution has started")


wait_for_execution_to_start(model_bias_monitor)

In [ ]:
# stop the schedule: no further executions, but the ongoing one continues
model_bias_monitor.stop_monitoring_schedule()

In [ ]:
def wait_for_execution_to_finish(model_monitor):
    schedule_desc = model_monitor.describe_schedule()
    execution_summary = schedule_desc.get("LastMonitoringExecutionSummary")
    if execution_summary is not None:
        print("Waiting for execution to finish", end="")
        while execution_summary["MonitoringExecutionStatus"] not in [
            "Completed",
            "CompletedWithViolations",
            "Failed",
            "Stopped",
        ]:
            print(".", end="", flush=True)
            time.sleep(60)
            schedule_desc = model_monitor.describe_schedule()
            execution_summary = schedule_desc["LastMonitoringExecutionSummary"]
        print("\nDone! Execution has finished")
    else:
        print("Last execution not found")


wait_for_execution_to_finish(model_bias_monitor)

### 📸 Monitor report output — screenshot this cell's output for submission

Lists the report files the Clarify monitor produced for the latest execution and prints the violations (bias metrics that drifted outside the baseline bounds, if any). The same results are visualized in **SageMaker Studio → Endpoints → (this endpoint) → Monitoring tab** — a screenshot of either view satisfies the assignment.

In [ ]:
schedule_desc = model_bias_monitor.describe_schedule()
execution_summary = schedule_desc.get("LastMonitoringExecutionSummary")
if execution_summary and execution_summary["MonitoringExecutionStatus"] in [
    "Completed",
    "CompletedWithViolations",
]:
    last_execution = model_bias_monitor.list_executions()[-1]
    report_uri = last_execution.output.destination
    print(f"Report URI: {report_uri}")
    report_files = sorted(S3Downloader.list(report_uri))
    print("Found report files:")
    print("\n ".join(report_files))

    violations = last_execution.constraint_violations()
    if violations:
        print("\nViolations (bias metrics outside baseline constraints):")
        print(json.dumps(violations.body_dict, indent=2))
    else:
        print("\nNo constraint violations: bias metrics are within baseline bounds.")

    # analysis.json holds the computed bias metrics (CI, baseline, current)
    analysis_files = [f for f in report_files if f.endswith("analysis.json")]
    if analysis_files:
        analysis = json.loads(S3Downloader.read_file(analysis_files[0]))
        print("\nBias metrics from analysis.json:")
        print(json.dumps(analysis, indent=2)[:4000])
else:
    print("====STOP==== No completed execution to inspect; check the schedule in the SageMaker console.")

---
## Section 7 — Cleanup (run after screenshotting the report)

Stop the worker threads, delete the monitoring schedule, then delete the endpoint and model to stop charges. Captured data, ground truth, baselines, and reports remain in S3.

In [ ]:
invoke_endpoint_thread.terminate()
ground_truth_thread.terminate()

predictor = Predictor(endpoint_name, sagemaker_session=sagemaker_session)
for model_monitor in predictor.list_monitors():
    model_monitor.stop_monitoring_schedule()
    model_monitor.delete_monitoring_schedule()

predictor.delete_endpoint()
predictor.delete_model()
print("Cleanup complete: threads stopped, schedule deleted, endpoint and model deleted.")

---
## Submission checklist

1. 📸 Screenshot the **Section 6 report cell output** (or the Studio endpoint monitoring view).
2. **File → Save As** the notebook as `Sourangshu_Pal_Assignment5.1`.
3. Export the executed notebook to PDF: download the `.ipynb`, then locally `jupyter nbconvert --to html Sourangshu_Pal_Assignment5.1.ipynb` → open the HTML in a browser → Print → Save as PDF (no LaTeX needed).
4. Submit **`Sourangshu_Pal_Assignment5.1.ipynb`** and **`Sourangshu_Pal_Assignment5.1.pdf`** to Canvas (due Oct 6).